In [1]:
"""
FLEMMS PCA Feature Selection Analysis
======================================
Individual-level microdata analysis with merged RTF1, RTF2, RTF3 datasets

Author: Catan, Ibo
Purpose: Dimensionality reduction and feature selection for predictive modeling of all FLEMMS Variables
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## FLEMMS Datasets Overview

In [5]:
rtf1_file = r"data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - RTF1.CSV"
rtf2_file = r"data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - RTF2.CSV"
rtf3_file = r"data/PHL-PSA-FLEMMS-2024-V2-PUF/FLEMMS PUF 2024 Volume2 - RTF3.CSV"
member_file = r"data/PHL-PSA-FLEMMS-2024-V1-PUF/FLEMMS PUF 2024 Volume1 - MEMBER.CSV"

df1 = pd.read_csv(rtf1_file)
df2 = pd.read_csv(rtf2_file)
df3 = pd.read_csv(rtf3_file)
df_mem = pd.read_csv(member_file)


In [6]:
df1.columns

Index(['REG', 'REG2', 'PRV', 'HHID', 'URBANITY', 'PSU_F1', 'BENEFICIARY_4PS', 'CHILDREN_04', 'ADULTS', 'GUARDIAN', 'CHILD_FACILITY', 'BUILDING', 'ROOF', 'WALL', 'FLOOR_FINISH', 'FLOOR_MAIN', 'TENURE', 'ELECTRICITY', 'HH_REF', 'HH_AIRCON', 'HH_WASHING', 'HH_OVEN', 'HH_RADIO', 'HH_ANATV', 'HH_DIGITV', 'HH_AUDIO', 'HH_LANDLINE', 'HH_BASICPHONE', 'HH_SMARTPHONE', 'HH_TABLET', 'HH_PC', 'HH_CAR', 'HH_VAN', 'HH_JEEP', 'HH_TRUCK', 'HH_MOTOR', 'HH_EBIKE', 'HH_TRICYCLE', 'HH_BIKE', 'HH_PEDICAB', 'HH_MBOAT', 'HH_NBOAT', 'HH_TRACTOR', 'HH_ANIMAL', 'HH_CABLE', 'HH_CABNET', 'HH_NOCABNET', 'HH_MOBSUB', 'HH_VSTREAM', 'HH_MSTREAM', 'DRINK_WATER', 'OTHERS_WATER', 'TOILET', 'TOILET_LOC', 'TOILET_SHARE', 'TOILET_PUBLIC', 'ODL_FAM', 'ICT_EQUIP', 'ODL_SOFTWARE', 'ODL_TECHNOLOGY', 'HH_RFACT'], dtype='object')

In [51]:
df2.columns

Index(['REG', 'REG2', 'PRV', 'HHID', 'URBANITY', 'PSU_F2', 'LNO_F2',
       'RESULT_CODE_F2', 'RESP_AGREE_F2', 'READ_IND_F2', 'AGE_F2', 'SEX_F2',
       'REL_F2', 'HGC_GRADE_F2', 'READING59', 'WRITING59', 'COMPUTE59',
       'LLEVEL59', 'READING', 'WRITING', 'COMPUTE', 'COMPRE', 'LLEVEL10OVER',
       'LLEVEL5OVER', 'FLLEVEL', 'BLITERATE', 'FLITERATE', 'RESP_RFACT_F2'],
      dtype='object')

In [7]:
df3.columns

Index(['REG', 'REG2', 'PRV', 'HHID', 'URBANITY', 'PSU_F3', 'LNO_F3', 'RESULT_CODE_F3', 'RESP_AGREE_F3', 'READ_IND_F3', 'MM_PNPAPER', 'MM_OLNPAPER', 'MM_MAGAZINE', 'MM_POSTERS', 'MM_TV', 'MM_RADIO', 'MM_MOVIES', 'MM_VSTREAM', 'MM_MSTREAM', 'MM_WORK', 'MM_SOCMED', 'MM_ECOMMERCE', 'MM_GSERV', 'MM_MEETINGS', 'MM_REPORTS', 'MM_CALCULATE', 'MM_CONTENT', 'INTERNET', 'DSKILL_INFO1', 'DSKILL_INFO2', 'DSKILL_INFO3', 'DSKILL_INFO4', 'DSKILL_COM1', 'DSKILL_COM2', 'DSKILL_COM3', 'DSKILL_COM4', 'DSKILL_CONT1', 'DSKILL_CONT2', 'DSKILL_CONT3', 'DSKILL_CONT4', 'DSKILL_CONT5', 'DSKILL_CONT6', 'DSKILL_SAFE1', 'DSKILL_SAFE2', 'DSKILL_PROB1', 'DSKILL_PROB2', 'DSKILL_PROB3', 'DSKILL_PROB4', 'DSKILL_PROB5', 'DSKILL_PROB6', 'TVOCGRAD', 'TVOCCOURSE', 'TVOCYEAR', 'TVOCBENE', 'TVOCAT', 'TVOCCERT', 'TVOCYAT', 'TVOCNOAT', 'TVOCTRAIN', 'TVOCASST', 'PATHWAY_COLLEGE', 'PATHWAY_SKILLS', 'PATHWAY_BUSINESS', 'PATHWAY_EMPLOYMENT', 'PATHWAY_STUDYABROAD', 'PATHWAY_WORKABROAD', 'PATHWAY_OTHERS', 'SPORTS', 'MSPORT',
       '

In [53]:
df_mem.columns

Index(['REG', 'REG2', 'PRV', 'HHID', 'URBANITY', 'PSU_MEM', 'LNO', 'SEX',
       'AGE', 'REL', 'MOTHER', 'MOTHER_LNO', 'FATHER', 'FATHER_LNO',
       'ETHNICITY', 'MSTAT', 'OF', 'OF_PRESENT', 'ATTEND_SCHOOL',
       'ATTEND_GRADE_LEVEL', 'ATTEND_GRADE', 'MODETRVL1', 'MODETRVL2',
       'MODETRVL3', 'NATT_REASON', 'BASIC_LIT', 'BASIC_NUM', 'HGC_LEVEL',
       'HGC_GRADE', 'DAYCARE', 'KINDER', 'SHS', 'SHS_TRACK', 'ALS', 'WORK',
       'WORK_TYPE_MAJOR', 'WORK_CLASS', 'NOWORK_REASON', 'DIFF_SEEING',
       'DIFF_HEARING', 'DIFF_WALKING', 'DIFF_REMEMBERING', 'DIFF_SELFCARE',
       'DIFF_COMMUNICATING', 'MEM_RFACT'],
      dtype='object')

## Feature Selection for PCA

## Helper Function

In [8]:
def standardize_column_names(df, dataset_name=""):
    """
    Standardize column names: lowercase and clean
    
    CRITICAL: Returns ONLY the DataFrame, not a tuple!
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    dataset_name : str
        Name of dataset for logging
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with standardized column names (NOT A TUPLE!)
    """
    original_cols = df.columns.tolist()
    
    # Lowercase all column names
    df.columns = df.columns.str.lower()
    
    # Replace spaces with underscores
    df.columns = df.columns.str.replace(' ', '_', regex=False)
    
    # Remove special characters (keep only alphanumeric and underscore)
    df.columns = df.columns.str.replace(r'[^a-z0-9_]', '', regex=True)
    
    standardized_cols = df.columns.tolist()
    
    # Log changes
    changes = [(orig, std) for orig, std in zip(original_cols, standardized_cols) 
               if orig != std]
    
    if changes and dataset_name:
        print(f"  📝 Standardized {len(changes)} column names in {dataset_name}:")
        for orig, std in changes[:5]:
            print(f"      '{orig}' → '{std}'")
        if len(changes) > 5:
            print(f"      ... and {len(changes) - 5} more")
    
    # IMPORTANT: Return ONLY the DataFrame
    return df

In [9]:
# 1. VARIABLES TO EXCLUDE (as specified by user)
EXCLUDED_REGIONS = ['reg', 'reg2']  # REG and REG2 as requested

# 2. DATA LEAKAGE VARIABLES (RTF2 test scores - these DEFINE functional literacy)
LEAKAGE_VARS = [
    # Test scores that define literacy
    'reading', 'writing', 'compute', 'compre',  # 10+ years old
    'reading59', 'writing59', 'compute59',      # 5-9 years old
    
    # Literacy level classifications (derived from test scores)
    'llevel59', 'llevel10over', 'llevel5over', 'fllevel',
    
    # Binary literacy outcomes (TARGET VARIABLE - don't use as predictor!)
    'bliterate',   # Basic literacy
    'fliterate',   # Functional literacy (THIS IS YOUR Y VARIABLE!)
    
    # Reading indicators (may be test-based)
    'read_ind_f2', 'read_ind_f3'
]

# 3. SURVEY DESIGN VARIABLES (not meaningful for PCA)
SURVEY_DESIGN_VARS = [
    # Identifiers
    'hhid', 'lno', 'lno_f2', 'lno_f3',
    
    # PSU (Primary Sampling Unit)
    'psu_mem', 'psu_f1', 'psu_f2', 'psu_f3',
    
    # Weights (don't include in PCA, but we'll use them later)
    'mem_rfact', 'hh_rfact', 'resp_rfact_f2', 'resp_rfact_f3',
    
    # Result codes (survey quality indicators)
    'result_code_f2', 'result_code_f3',
    
    # Response agreement (survey process)
    'resp_agree_f2', 'resp_agree_f3',
    
    # Mother/Father identifiers (relational, not attributes)
    'mother', 'mother_lno', 'father', 'father_lno'
]

# 4. GEOGRAPHIC IDENTIFIERS (keep for grouping but exclude from PCA)
GEOGRAPHIC_VARS = [
    'prv'
]

# 5. DUPLICATE DEMOGRAPHIC VARIABLES (appear in multiple RTFs)
# Keep only one version to avoid perfect multicollinearity

DUPLICATE_VARS = [
    'age_f2', 'age_f3',      # Keep 'age' from MEMBER
    'sex_f2', 'sex_f3',      # Keep 'sex' from MEMBER
    'rel_f2', 'rel_f3',      # Keep 'rel' from MEMBER
    'hgc_grade_f2', 'hgc_grade_f3'  # Keep 'hgc_grade' from MEMBER
    'prv_rtf1','prv_rtf2','prv_rtf3',
    'urbanity_rtf3','urbanity_rtf2'
]

# 6. CUSTOMIZABLE EXCLUSION LIST - ADD YOUR OWN HERE
cols_to_remove = [

    # Add any other variables you want to exclude here
]

In [10]:
# ============================================================================
# MERGE ALL EXCLUSION LISTS
# ============================================================================

ALL_EXCLUDED_VARS = (
    EXCLUDED_REGIONS + 
    LEAKAGE_VARS + 
    SURVEY_DESIGN_VARS + 
    DUPLICATE_VARS + 
    cols_to_remove
)

In [11]:
# Remove 'prv' from exclusion if you want to use it for grouping
# but we'll handle it separately below
ALL_EXCLUDED_VARS = [v for v in ALL_EXCLUDED_VARS if v != 'prv']

print("="*80)
print("FLEMMS PCA FEATURE SELECTION ANALYSIS")
print("="*80)
print(f"\nTotal variables to exclude: {len(ALL_EXCLUDED_VARS)}")
print("\nExclusion categories:")
print(f"  - Region variables (REG, REG2): {len(EXCLUDED_REGIONS)}")
print(f"  - Data leakage (test scores): {len(LEAKAGE_VARS)}")
print(f"  - Survey design variables: {len(SURVEY_DESIGN_VARS)}")
print(f"  - Duplicate demographics: {len(DUPLICATE_VARS)}")
print(f"  - Custom exclusions: {len(cols_to_remove)}")
print("\n" + "="*80)


# ============================================================================
# DATA LOADING
# ============================================================================

def load_flemms_data(rtf1_file, rtf2_file, rtf3_file, member_file):
    """
    Load and merge all FLEMMS datasets
    
    Parameters:
    -----------
    rtf1_file : str
        Path to RTF1 CSV (household questions)
    rtf2_file : str
        Path to RTF2 CSV (literacy indicators)
    rtf3_file : str
        Path to RTF3 CSV (mass media indicators)
    member_file : str
        Path to MEMBER CSV (member record)
    
    Returns:
    --------
    df_merged : pd.DataFrame
        Merged dataset
    """
    
    print("\n[1/5] Loading datasets...")
    
    df1 = pd.read_csv(rtf1_file)
    df2 = pd.read_csv(rtf2_file)
    df3 = pd.read_csv(rtf3_file)
    df_mem = pd.read_csv(member_file)

    df1 = standardize_column_names(df1, "RTF1")
    df2 = standardize_column_names(df2, "RTF2")
    df3 = standardize_column_names(df3, "RTF3")
    df_mem = standardize_column_names(df_mem, "MEMBER")
    
    print(f"  ✓ RTF1 loaded: {df1.shape}")
    print(f"  ✓ RTF2 loaded: {df2.shape}")
    print(f"  ✓ RTF3 loaded: {df3.shape}")
    print(f"  ✓ MEMBER loaded: {df_mem.shape}")
    
    # Merge strategy:
    # 1. Start with MEMBER (most granular individual-level data)
    # 2. Merge RTF1 (household-level) on hhid
    # 3. Merge RTF2 (individual literacy) on hhid + lno_f2
    # 4. Merge RTF3 (individual mass media) on hhid + lno_f3
    
    print("\n[2/5] Merging datasets...")
    
    # Merge MEMBER with RTF1 (household level)
    df_merged = df_mem.merge(df1, on='hhid', how='left', suffixes=('', '_rtf1'))
    print(f"  ✓ After MEMBER + RTF1: {df_merged.shape}")
    
    # Merge with RTF2 (individual literacy data)
    # RTF2 uses lno_f2, MEMBER uses lno
    df2_renamed = df2.rename(columns={'lno_f2': 'lno'})
    df_merged = df_merged.merge(df2_renamed, on=['hhid', 'lno'], how='left', suffixes=('', '_rtf2'))
    print(f"  ✓ After + RTF2: {df_merged.shape}")
    
    # Merge with RTF3 (individual mass media data)
    # RTF3 uses lno_f3, MEMBER uses lno
    df3_renamed = df3.rename(columns={'lno_f3': 'lno'})
    df_merged = df_merged.merge(df3_renamed, on=['hhid', 'lno'], how='left', suffixes=('', '_rtf3'))
    print(f"  ✓ After + RTF3: {df_merged.shape}")
    
    return df_merged


# ============================================================================
# DATA PREPROCESSING
# ============================================================================

def preprocess_for_pca(df, excluded_vars):
    """
    Prepare data for PCA analysis
    
    Parameters:
    -----------
    df : pd.DataFrame
        Merged FLEMMS dataset
    excluded_vars : list
        Variables to exclude from analysis
    
    Returns:
    --------
    X_scaled : np.ndarray
        Standardized feature matrix
    feature_names : list
        Names of features included in PCA
    df_metadata : pd.DataFrame
        Metadata columns (identifiers, weights, target variable)
    """
    
    print("\n[3/5] Preprocessing data for PCA...")
    
    # Identify columns to keep for metadata (not in PCA, but important)
    metadata_cols = ['hhid', 'lno', 'prv', 'urbanity', 'fliterate', 
                     'mem_rfact', 'hh_rfact', 'resp_rfact_f2']
    metadata_cols = [c for c in metadata_cols if c in df.columns]
    
    df_metadata = df[metadata_cols].copy()
    
    # Remove excluded variables
    cols_to_drop = [c for c in excluded_vars if c in df.columns]
    cols_to_drop += metadata_cols  # Also drop metadata from feature matrix
    
    df_features = df.drop(columns=cols_to_drop, errors='ignore')
    
    print(f"  ✓ Dropped {len(cols_to_drop)} excluded/metadata columns")
    print(f"  ✓ Remaining columns: {df_features.shape[1]}")
    
    # Separate numeric and categorical variables
    numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_features.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"\n  Variable types:")
    print(f"    - Numeric: {len(numeric_cols)}")
    print(f"    - Categorical: {len(categorical_cols)}")
    
    # One-hot encode categorical variables
    if len(categorical_cols) > 0:
        print(f"\n  One-hot encoding {len(categorical_cols)} categorical variables...")
        df_encoded = pd.get_dummies(df_features, columns=categorical_cols, 
                                     drop_first=True, dtype=int)
    else:
        df_encoded = df_features.copy()
    
    print(f"  ✓ After encoding: {df_encoded.shape[1]} features")
    
    # Handle missing values
    print(f"\n  Handling missing values...")
    missing_counts = df_encoded.isnull().sum()
    cols_with_missing = missing_counts[missing_counts > 0]
    
    if len(cols_with_missing) > 0:
        print(f"  ⚠ Found {len(cols_with_missing)} columns with missing values")
        print(f"    Total missing: {missing_counts.sum()} cells")
        
        # Impute with median for numeric variables
        imputer = SimpleImputer(strategy='median')
        df_imputed = pd.DataFrame(
            imputer.fit_transform(df_encoded),
            columns=df_encoded.columns,
            index=df_encoded.index
        )
    else:
        df_imputed = df_encoded.copy()
    
    # Standardize features (critical for PCA)
    print(f"\n  Standardizing features (z-score normalization)...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_imputed)
    
    feature_names = df_imputed.columns.tolist()
    
    print(f"  ✓ Final feature matrix: {X_scaled.shape}")
    print(f"    - Samples (individuals): {X_scaled.shape[0]:,}")
    print(f"    - Features: {X_scaled.shape[1]}")
    
    return X_scaled, feature_names, df_metadata


# ============================================================================
# PCA ANALYSIS
# ============================================================================

def perform_pca_analysis(X_scaled, feature_names, n_components=None, 
                         variance_threshold=0.95):
    """
    Perform PCA and generate comprehensive results
    
    Parameters:
    -----------
    X_scaled : np.ndarray
        Standardized feature matrix
    feature_names : list
        Feature names
    n_components : int, optional
        Number of components (default: automatic based on variance threshold)
    variance_threshold : float
        Cumulative variance threshold for automatic component selection
    
    Returns:
    --------
    pca : PCA object
        Fitted PCA model
    X_pca : np.ndarray
        Transformed data
    results : dict
        Dictionary containing analysis results
    """
    
    print("\n[4/5] Performing PCA...")
    
    # If n_components not specified, use all components first
    if n_components is None:
        pca_full = PCA()
        pca_full.fit(X_scaled)
        
        # Find number of components needed for variance threshold
        cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)
        n_components = np.argmax(cumsum_var >= variance_threshold) + 1
        
        print(f"  ℹ Auto-selected {n_components} components " +
              f"(explaining {variance_threshold*100}% variance)")
    
    # Fit PCA with selected number of components
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    
    print(f"  ✓ PCA complete: {n_components} components")
    print(f"  ✓ Explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    
    # Create results dictionary
    results = {
        'n_components': n_components,
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
        'loadings': pca.components_,
        'feature_names': feature_names,
        'X_pca': X_pca
    }
    
    # Create loadings DataFrame
    loadings_df = pd.DataFrame(
        pca.components_.T,
        columns=[f'PC{i+1}' for i in range(n_components)],
        index=feature_names
    )
    results['loadings_df'] = loadings_df
    
    return pca, X_pca, results


# ============================================================================
# VISUALIZATION
# ============================================================================

def create_visualizations(pca, results, output_dir='./pca_results'):
    """
    Create comprehensive PCA visualizations
    
    Parameters:
    -----------
    pca : PCA object
        Fitted PCA model
    results : dict
        Results from PCA analysis
    output_dir : str
        Directory to save plots
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n[5/5] Creating visualizations...")
    print(f"  Output directory: {output_dir}")
    
    # Set style
    plt.style.use('ggplot')
    sns.set_palette("husl")
    
    # -------------------------------------------------------------------------
    # 1. SCREE PLOT (Explained Variance)
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Individual variance
    axes[0].bar(range(1, len(results['explained_variance_ratio'])+1),
                results['explained_variance_ratio'],
                alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Principal Component', fontsize=12)
    axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
    axes[0].set_title('Scree Plot: Individual Variance Explained', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Cumulative variance
    axes[1].plot(range(1, len(results['cumulative_variance'])+1),
                 results['cumulative_variance'],
                 marker='o', linewidth=2, markersize=6, color='darkred')
    axes[1].axhline(y=0.95, color='green', linestyle='--', label='95% threshold')
    axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
    axes[1].set_xlabel('Number of Components', fontsize=12)
    axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
    axes[1].set_title('Cumulative Variance Explained', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/01_scree_plot.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: 01_scree_plot.png")
    plt.close()
    
    # -------------------------------------------------------------------------
    # 2. LOADINGS HEATMAP (Top 20 features for first 5 PCs)
    # -------------------------------------------------------------------------
    n_pcs_to_show = min(5, results['n_components'])
    loadings_df = results['loadings_df'].iloc[:, :n_pcs_to_show]
    
    # Get top features by absolute loading
    top_features_per_pc = []
    for pc in loadings_df.columns:
        top_idx = loadings_df[pc].abs().nlargest(20).index
        top_features_per_pc.extend(top_idx)
    
    top_features = list(set(top_features_per_pc))[:50]  # Max 50 features
    loadings_subset = loadings_df.loc[top_features]
    
    fig, ax = plt.subplots(figsize=(12, max(10, len(top_features)*0.3)))
    sns.heatmap(loadings_subset, cmap='RdBu_r', center=0, 
                cbar_kws={'label': 'Loading Value'},
                linewidths=0.5, ax=ax)
    ax.set_title(f'Feature Loadings Heatmap (Top {len(top_features)} Features)',
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Principal Component', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/02_loadings_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: 02_loadings_heatmap.png")
    plt.close()
    
    # -------------------------------------------------------------------------
    # 3. BIPLOT (PC1 vs PC2 with feature vectors)
    # -------------------------------------------------------------------------
    if results['n_components'] >= 2:
        fig, ax = plt.subplots(figsize=(14, 10))
        
        # Plot transformed data (sample with max 5000 points for visibility)
        n_samples = min(5000, results['X_pca'].shape[0])
        sample_idx = np.random.choice(results['X_pca'].shape[0], n_samples, replace=False)
        
        ax.scatter(results['X_pca'][sample_idx, 0], 
                   results['X_pca'][sample_idx, 1],
                   alpha=0.3, s=10, color='gray', label='Observations')
        
        # Plot feature vectors (top 15 by length)
        loadings_pc1_pc2 = results['loadings'][:2, :].T
        loading_lengths = np.sqrt((loadings_pc1_pc2**2).sum(axis=1))
        top_15_idx = loading_lengths.argsort()[-15:]
        
        for idx in top_15_idx:
            ax.arrow(0, 0, 
                     loadings_pc1_pc2[idx, 0]*3, 
                     loadings_pc1_pc2[idx, 1]*3,
                     head_width=0.1, head_length=0.1, 
                     fc='red', ec='red', alpha=0.7)
            ax.text(loadings_pc1_pc2[idx, 0]*3.2, 
                    loadings_pc1_pc2[idx, 1]*3.2,
                    results['feature_names'][idx],
                    fontsize=9, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
        
        ax.set_xlabel(f"PC1 ({results['explained_variance_ratio'][0]*100:.1f}% variance)",
                      fontsize=12)
        ax.set_ylabel(f"PC2 ({results['explained_variance_ratio'][1]*100:.1f}% variance)",
                      fontsize=12)
        ax.set_title('PCA Biplot: PC1 vs PC2 with Feature Vectors',
                     fontsize=14, fontweight='bold')
        ax.axhline(y=0, color='k', linewidth=0.5, linestyle='--', alpha=0.3)
        ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--', alpha=0.3)
        ax.grid(alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/03_biplot_pc1_pc2.png', dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: 03_biplot_pc1_pc2.png")
        plt.close()
    
    print(f"\n  ✓ All visualizations saved to: {output_dir}/")


# ============================================================================
# FEATURE SELECTION BASED ON PCA
# ============================================================================

def select_features_from_pca(results, n_components_to_use=None, 
                              top_n_per_component=10):
    """
    Select most important features based on PCA loadings
    
    Parameters:
    -----------
    results : dict
        PCA results
    n_components_to_use : int, optional
        Number of components to consider (default: all)
    top_n_per_component : int
        Number of top features to select per component
    
    Returns:
    --------
    selected_features : list
        List of selected feature names
    feature_importance_df : pd.DataFrame
        DataFrame with feature importance scores
    """
    
    print("\n" + "="*80)
    print("FEATURE SELECTION RESULTS")
    print("="*80)
    
    if n_components_to_use is None:
        n_components_to_use = results['n_components']
    
    loadings_df = results['loadings_df'].iloc[:, :n_components_to_use]
    
    # Calculate overall importance as sum of absolute loadings across components
    feature_importance = loadings_df.abs().sum(axis=1).sort_values(ascending=False)
    
    # Also calculate max absolute loading (most influential in any single PC)
    max_loading = loadings_df.abs().max(axis=1).sort_values(ascending=False)
    
    # Combine into DataFrame
    feature_importance_df = pd.DataFrame({
        'Feature': feature_importance.index,
        'Total_Importance': feature_importance.values,
        'Max_Loading': max_loading.loc[feature_importance.index].values
    }).reset_index(drop=True)
    
    # Select top features per component
    selected_features_set = set()
    
    for i in range(n_components_to_use):
        pc_name = f'PC{i+1}'
        top_features = loadings_df[pc_name].abs().nlargest(top_n_per_component).index.tolist()
        selected_features_set.update(top_features)
        
        print(f"\n{pc_name} - Top {top_n_per_component} features:")
        for j, feat in enumerate(top_features, 1):
            loading = loadings_df.loc[feat, pc_name]
            print(f"  {j:2d}. {feat:40s} (loading: {loading:7.4f})")
    
    selected_features = sorted(list(selected_features_set))
    
    print("\n" + "="*80)
    print(f"TOTAL SELECTED FEATURES: {len(selected_features)}")
    print("="*80)
    
    return selected_features, feature_importance_df


# ============================================================================
# EXPORT RESULTS
# ============================================================================

def export_results(results, feature_importance_df, selected_features, 
                   output_dir='./pca_results'):
    """
    Export PCA results to CSV files
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\nExporting results to: {output_dir}/")
    
    # 1. Explained variance
    variance_df = pd.DataFrame({
        'Component': [f'PC{i+1}' for i in range(results['n_components'])],
        'Explained_Variance_Ratio': results['explained_variance_ratio'],
        'Cumulative_Variance': results['cumulative_variance']
    })
    variance_df.to_csv(f'{output_dir}/explained_variance.csv', index=False)
    print(f"  ✓ explained_variance.csv")
    
    # 2. Full loadings matrix
    results['loadings_df'].to_csv(f'{output_dir}/loadings_matrix.csv')
    print(f"  ✓ loadings_matrix.csv")
    
    # 3. Feature importance
    feature_importance_df.to_csv(f'{output_dir}/feature_importance.csv', index=False)
    print(f"  ✓ feature_importance.csv")
    
    # 4. Selected features
    selected_df = pd.DataFrame({'Selected_Features': selected_features})
    selected_df.to_csv(f'{output_dir}/selected_features.csv', index=False)
    print(f"  ✓ selected_features.csv")
    
    # 5. Summary report
    with open(f'{output_dir}/pca_summary_report.txt', 'w') as f:
        f.write("="*80 + "\n")
        f.write("FLEMMS PCA ANALYSIS SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Number of components: {results['n_components']}\n")
        f.write(f"Total explained variance: {results['cumulative_variance'][-1]*100:.2f}%\n")
        f.write(f"Number of original features: {len(results['feature_names'])}\n")
        f.write(f"Number of selected features: {len(selected_features)}\n")
        f.write(f"Reduction rate: {(1 - len(selected_features)/len(results['feature_names']))*100:.1f}%\n\n")
        
        f.write("Variance by component:\n")
        for i, var in enumerate(results['explained_variance_ratio'][:10], 1):
            f.write(f"  PC{i}: {var*100:.2f}%\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("Top 20 Most Important Features (by total loading):\n")
        f.write("="*80 + "\n")
        for i, row in feature_importance_df.head(20).iterrows():
            f.write(f"{i+1:3d}. {row['Feature']:45s} " +
                    f"(importance: {row['Total_Importance']:.4f})\n")
    
    print(f"  ✓ pca_summary_report.txt")
    print(f"\n✅ All results exported successfully!")


# ============================================================================
# MAIN EXECUTION FUNCTION
# ============================================================================

def run_flemms_pca(rtf1_file, rtf2_file, rtf3_file, member_file,
                   n_components=None, variance_threshold=0.95,
                   top_n_per_component=10, output_dir='./pca_results'):
    """
    Main function to run complete PCA analysis pipeline
    
    Parameters:
    -----------
    rtf1_file, rtf2_file, rtf3_file, member_file : str
        Paths to FLEMMS CSV files
    n_components : int, optional
        Number of PCA components (default: auto-select based on variance)
    variance_threshold : float
        Cumulative variance threshold for auto-selection
    top_n_per_component : int
        Number of top features to select per component
    output_dir : str
        Output directory for results
    
    Returns:
    --------
    results : dict
        Complete analysis results
    selected_features : list
        List of selected features for modeling
    """
    
    # Load data
    df_merged = load_flemms_data(rtf1_file, rtf2_file, rtf3_file, member_file)
    
    # Preprocess
    X_scaled, feature_names, df_metadata = preprocess_for_pca(
        df_merged, ALL_EXCLUDED_VARS
    )
    
    # Perform PCA
    pca, X_pca, results = perform_pca_analysis(
        X_scaled, feature_names, n_components, variance_threshold
    )
    
    # Visualize
    create_visualizations(pca, results, output_dir)
    
    # Feature selection
    selected_features, feature_importance_df = select_features_from_pca(
        results, n_components_to_use=None, top_n_per_component=top_n_per_component
    )
    
    # Export results
    export_results(results, feature_importance_df, selected_features, output_dir)
    
    # Add metadata to results
    results['df_metadata'] = df_metadata
    results['selected_features'] = selected_features
    results['feature_importance_df'] = feature_importance_df
    
    return results, selected_features


# ============================================================================
# USAGE EXAMPLE
# ============================================================================

FLEMMS PCA FEATURE SELECTION ANALYSIS

Total variables to exclude: 49

Exclusion categories:
  - Region variables (REG, REG2): 2
  - Data leakage (test scores): 15
  - Survey design variables: 20
  - Duplicate demographics: 12
  - Custom exclusions: 0



## Driver Code

In [12]:
# ============================================================================
# DATA LOADING
# ============================================================================


def load_flemms_data(rtf1_file, rtf2_file, rtf3_file, member_file):
    """
    Load and merge all FLEMMS datasets
    
    Parameters:
    -----------
    rtf1_file : str
        Path to RTF1 CSV (household questions)
    rtf2_file : str
        Path to RTF2 CSV (literacy indicators)
    rtf3_file : str
        Path to RTF3 CSV (mass media indicators)
    member_file : str
        Path to MEMBER CSV (member record)
    
    Returns:
    --------
    df_merged : pd.DataFrame
        Merged dataset
    """
    
    print("\n[1/5] Loading datasets...")
    
    df1 = pd.read_csv(rtf1_file)
    df2 = pd.read_csv(rtf2_file)
    df3 = pd.read_csv(rtf3_file)
    df_mem = pd.read_csv(member_file)
    
    print(f"  ✓ RTF1 loaded: {df1.shape}")
    print(f"  ✓ RTF2 loaded: {df2.shape}")
    print(f"  ✓ RTF3 loaded: {df3.shape}")
    print(f"  ✓ MEMBER loaded: {df_mem.shape}")
    
    # Standardize column names (lowercase)
    print("\n  Standardizing column names to lowercase...")
    df1 = standardize_column_names(df1, "RTF1")
    df2 = standardize_column_names(df2, "RTF2")
    df3 = standardize_column_names(df3, "RTF3")
    df_mem = standardize_column_names(df_mem, "MEMBER")
    
    # Merge strategy:
    # 1. Start with MEMBER (most granular individual-level data)
    # 2. Merge RTF1 (household-level) on hhid
    # 3. Merge RTF2 (individual literacy) on hhid + lno
    # 4. Merge RTF3 (individual mass media) on hhid + lno
    
    print("\n[2/5] Merging datasets...")
    
    # Check if hhid exists in all datasets
    for df, name in [(df_mem, 'MEMBER'), (df1, 'RTF1'), (df2, 'RTF2'), (df3, 'RTF3')]:
        if 'hhid' not in df.columns:
            raise ValueError(f"{name} is missing 'hhid' column. Available columns: {df.columns.tolist()}")
    
    # Merge MEMBER with RTF1 (household level)
    df_merged = df_mem.merge(df1, on='hhid', how='left', suffixes=('', '_rtf1'))
    print(f"  ✓ After MEMBER + RTF1: {df_merged.shape}")
    
    # Merge with RTF2 (individual literacy data)
    # Check which line number column RTF2 has
    if 'lno_f2' in df2.columns:
        merge_cols_rtf2 = ['hhid', 'lno_f2']
        # Create matching column in df_merged if needed
        if 'lno_f2' not in df_merged.columns and 'lno' in df_merged.columns:
            df_merged['lno_f2'] = df_merged['lno']
    elif 'lno' in df2.columns:
        merge_cols_rtf2 = ['hhid', 'lno']
    else:
        # If no line number, merge only on hhid
        merge_cols_rtf2 = ['hhid']
        print("  ⚠ Warning: RTF2 has no lno column, merging on hhid only")
    
    df_merged = df_merged.merge(df2, on=merge_cols_rtf2, how='left', suffixes=('', '_rtf2'))
    print(f"  ✓ After + RTF2: {df_merged.shape}")
    
    # Merge with RTF3 (individual mass media data)
    # Check which line number column RTF3 has
    if 'lno_f3' in df3.columns:
        merge_cols_rtf3 = ['hhid', 'lno_f3']
        # Create matching column in df_merged if needed
        if 'lno_f3' not in df_merged.columns and 'lno' in df_merged.columns:
            df_merged['lno_f3'] = df_merged['lno']
    elif 'lno' in df3.columns:
        merge_cols_rtf3 = ['hhid', 'lno']
    else:
        # If no line number, merge only on hhid
        merge_cols_rtf3 = ['hhid']
        print("  ⚠ Warning: RTF3 has no lno column, merging on hhid only")
    
    df_merged = df_merged.merge(df3, on=merge_cols_rtf3, how='left', suffixes=('', '_rtf3'))
    print(f"  ✓ After + RTF3: {df_merged.shape}")
    
    print(f"\n  Final merged dataset: {df_merged.shape}")
    print(f"  Total columns: {len(df_merged.columns)}")
    
    return df_merged


# ============================================================================
# DATA PREPROCESSING
# ============================================================================

def preprocess_for_pca(df, excluded_vars):
    """
    Prepare data for PCA analysis
    
    Parameters:
    -----------
    df : pd.DataFrame
        Merged FLEMMS dataset
    excluded_vars : list
        Variables to exclude from analysis
    
    Returns:
    --------
    X_scaled : np.ndarray
        Standardized feature matrix
    feature_names : list
        Names of features included in PCA
    df_metadata : pd.DataFrame
        Metadata columns (identifiers, weights, target variable)
    """
    
    print("\n[3/5] Preprocessing data for PCA...")
    
    # Validate input
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Expected pandas DataFrame, got {type(df)}. "
                       f"This usually means the merge operation returned the wrong type.")
    
    if df.shape[0] == 0:
        raise ValueError("DataFrame is empty! Check your data files.")
    
    print(f"  Input shape: {df.shape}")
    print(f"  Columns: {len(df.columns)}")
    
    # Identify columns to keep for metadata (not in PCA, but important)
    metadata_cols = ['hhid', 'lno', 'prv', 'urbanity', 'fliterate', 
                     'mem_rfact', 'hh_rfact', 'resp_rfact_f2']
    metadata_cols = [c for c in metadata_cols if c in df.columns]
    
    df_metadata = df[metadata_cols].copy()
    
    # Remove excluded variables
    cols_to_drop = [c for c in excluded_vars if c in df.columns]
    cols_to_drop += metadata_cols  # Also drop metadata from feature matrix
    
    df_features = df.drop(columns=cols_to_drop, errors='ignore')
    
    print(f"  ✓ Dropped {len(cols_to_drop)} excluded/metadata columns")
    print(f"  ✓ Remaining columns: {df_features.shape[1]}")
    
    # Separate numeric and categorical variables
    numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df_features.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"\n  Variable types:")
    print(f"    - Numeric: {len(numeric_cols)}")
    print(f"    - Categorical: {len(categorical_cols)}")
    
    # One-hot encode categorical variables
    if len(categorical_cols) > 0:
        print(f"\n  One-hot encoding {len(categorical_cols)} categorical variables...")
        df_encoded = pd.get_dummies(df_features, columns=categorical_cols, 
                                     drop_first=True, dtype=int)
    else:
        df_encoded = df_features.copy()
    
    print(f"  ✓ After encoding: {df_encoded.shape[1]} features")
    
    # Handle missing values
    print(f"\n  Handling missing values...")
    missing_counts = df_encoded.isnull().sum()
    cols_with_missing = missing_counts[missing_counts > 0]
    
    if len(cols_with_missing) > 0:
        print(f"  ⚠ Found {len(cols_with_missing)} columns with missing values")
        print(f"    Total missing: {missing_counts.sum()} cells")
        
        # Impute with median for numeric variables
        imputer = SimpleImputer(strategy='median')
        df_imputed = pd.DataFrame(
            imputer.fit_transform(df_encoded),
            columns=df_encoded.columns,
            index=df_encoded.index
        )
    else:
        df_imputed = df_encoded.copy()
    
    # Standardize features (critical for PCA)
    print(f"\n  Standardizing features (z-score normalization)...")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_imputed)
    
    feature_names = df_imputed.columns.tolist()
    
    print(f"  ✓ Final feature matrix: {X_scaled.shape}")
    print(f"    - Samples (individuals): {X_scaled.shape[0]:,}")
    print(f"    - Features: {X_scaled.shape[1]}")
    
    return X_scaled, feature_names, df_metadata


# ============================================================================
# PCA ANALYSIS
# ============================================================================

def perform_pca_analysis(X_scaled, feature_names, n_components=None, 
                         variance_threshold=0.95):
    """
    Perform PCA and generate comprehensive results
    
    Parameters:
    -----------
    X_scaled : np.ndarray
        Standardized feature matrix
    feature_names : list
        Feature names
    n_components : int, optional
        Number of components (default: automatic based on variance threshold)
    variance_threshold : float
        Cumulative variance threshold for automatic component selection
    
    Returns:
    --------
    pca : PCA object
        Fitted PCA model
    X_pca : np.ndarray
        Transformed data
    results : dict
        Dictionary containing analysis results
    """
    
    print("\n[4/5] Performing PCA...")
    
    # If n_components not specified, use all components first
    if n_components is None:
        pca_full = PCA()
        pca_full.fit(X_scaled)
        
        # Find number of components needed for variance threshold
        cumsum_var = np.cumsum(pca_full.explained_variance_ratio_)
        n_components = np.argmax(cumsum_var >= variance_threshold) + 1
        
        print(f"  ℹ Auto-selected {n_components} components " +
              f"(explaining {variance_threshold*100}% variance)")
    
    # Fit PCA with selected number of components
    pca = PCA(n_components=n_components)
    X_pca = pca.fit_transform(X_scaled)
    
    print(f"  ✓ PCA complete: {n_components} components")
    print(f"  ✓ Explained variance: {pca.explained_variance_ratio_.sum()*100:.2f}%")
    
    # Create results dictionary
    results = {
        'n_components': n_components,
        'explained_variance_ratio': pca.explained_variance_ratio_,
        'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
        'loadings': pca.components_,
        'feature_names': feature_names,
        'X_pca': X_pca
    }
    
    # Create loadings DataFrame
    loadings_df = pd.DataFrame(
        pca.components_.T,
        columns=[f'PC{i+1}' for i in range(n_components)],
        index=feature_names
    )
    results['loadings_df'] = loadings_df
    
    return pca, X_pca, results


# ============================================================================
# VISUALIZATION
# ============================================================================

def create_visualizations(pca, results, output_dir='./pca_results'):
    """
    Create comprehensive PCA visualizations
    
    Parameters:
    -----------
    pca : PCA object
        Fitted PCA model
    results : dict
        Results from PCA analysis
    output_dir : str
        Directory to save plots
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n[5/5] Creating visualizations...")
    print(f"  Output directory: {output_dir}")
    
    # Set style
    plt.style.use('ggplot')
    sns.set_palette("husl")
    
    # -------------------------------------------------------------------------
    # 1. SCREE PLOT (Explained Variance)
    # -------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Individual variance
    axes[0].bar(range(1, len(results['explained_variance_ratio'])+1),
                results['explained_variance_ratio'],
                alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Principal Component', fontsize=12)
    axes[0].set_ylabel('Explained Variance Ratio', fontsize=12)
    axes[0].set_title('Scree Plot: Individual Variance Explained', fontsize=14, fontweight='bold')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Cumulative variance
    axes[1].plot(range(1, len(results['cumulative_variance'])+1),
                 results['cumulative_variance'],
                 marker='o', linewidth=2, markersize=6, color='darkred')
    axes[1].axhline(y=0.95, color='green', linestyle='--', label='95% threshold')
    axes[1].axhline(y=0.90, color='orange', linestyle='--', label='90% threshold')
    axes[1].set_xlabel('Number of Components', fontsize=12)
    axes[1].set_ylabel('Cumulative Explained Variance', fontsize=12)
    axes[1].set_title('Cumulative Variance Explained', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/01_scree_plot.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: 01_scree_plot.png")
    plt.close()
    
    # -------------------------------------------------------------------------
    # 2. LOADINGS HEATMAP (Top 20 features for first 5 PCs)
    # -------------------------------------------------------------------------
    n_pcs_to_show = min(5, results['n_components'])
    loadings_df = results['loadings_df'].iloc[:, :n_pcs_to_show]
    
    # Get top features by absolute loading
    top_features_per_pc = []
    for pc in loadings_df.columns:
        top_idx = loadings_df[pc].abs().nlargest(20).index
        top_features_per_pc.extend(top_idx)
    
    top_features = list(set(top_features_per_pc))[:50]  # Max 50 features
    loadings_subset = loadings_df.loc[top_features]
    
    fig, ax = plt.subplots(figsize=(12, max(10, len(top_features)*0.3)))
    sns.heatmap(loadings_subset, cmap='RdBu_r', center=0, 
                cbar_kws={'label': 'Loading Value'},
                linewidths=0.5, ax=ax)
    ax.set_title(f'Feature Loadings Heatmap (Top {len(top_features)} Features)',
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xlabel('Principal Component', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(f'{output_dir}/02_loadings_heatmap.png', dpi=300, bbox_inches='tight')
    print(f"  ✓ Saved: 02_loadings_heatmap.png")
    plt.close()
    
    # -------------------------------------------------------------------------
    # 3. BIPLOT (PC1 vs PC2 with feature vectors)
    # -------------------------------------------------------------------------
    if results['n_components'] >= 2:
        fig, ax = plt.subplots(figsize=(14, 10))
        
        # Plot transformed data (sample with max 5000 points for visibility)
        n_samples = min(5000, results['X_pca'].shape[0])
        sample_idx = np.random.choice(results['X_pca'].shape[0], n_samples, replace=False)
        
        ax.scatter(results['X_pca'][sample_idx, 0], 
                   results['X_pca'][sample_idx, 1],
                   alpha=0.3, s=10, color='gray', label='Observations')
        
        # Plot feature vectors (top 15 by length)
        loadings_pc1_pc2 = results['loadings'][:2, :].T
        loading_lengths = np.sqrt((loadings_pc1_pc2**2).sum(axis=1))
        top_15_idx = loading_lengths.argsort()[-15:]
        
        for idx in top_15_idx:
            ax.arrow(0, 0, 
                     loadings_pc1_pc2[idx, 0]*3, 
                     loadings_pc1_pc2[idx, 1]*3,
                     head_width=0.1, head_length=0.1, 
                     fc='red', ec='red', alpha=0.7)
            ax.text(loadings_pc1_pc2[idx, 0]*3.2, 
                    loadings_pc1_pc2[idx, 1]*3.2,
                    results['feature_names'][idx],
                    fontsize=9, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
        
        ax.set_xlabel(f"PC1 ({results['explained_variance_ratio'][0]*100:.1f}% variance)",
                      fontsize=12)
        ax.set_ylabel(f"PC2 ({results['explained_variance_ratio'][1]*100:.1f}% variance)",
                      fontsize=12)
        ax.set_title('PCA Biplot: PC1 vs PC2 with Feature Vectors',
                     fontsize=14, fontweight='bold')
        ax.axhline(y=0, color='k', linewidth=0.5, linestyle='--', alpha=0.3)
        ax.axvline(x=0, color='k', linewidth=0.5, linestyle='--', alpha=0.3)
        ax.grid(alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/03_biplot_pc1_pc2.png', dpi=300, bbox_inches='tight')
        print(f"  ✓ Saved: 03_biplot_pc1_pc2.png")
        plt.close()
    
    print(f"\n  ✓ All visualizations saved to: {output_dir}/")


# ============================================================================
# FEATURE SELECTION BASED ON PCA
# ============================================================================

def select_features_from_pca(results, n_components_to_use=None, 
                              top_n_per_component=10):
    """
    Select most important features based on PCA loadings
    
    Parameters:
    -----------
    results : dict
        PCA results
    n_components_to_use : int, optional
        Number of components to consider (default: all)
    top_n_per_component : int
        Number of top features to select per component
    
    Returns:
    --------
    selected_features : list
        List of selected feature names
    feature_importance_df : pd.DataFrame
        DataFrame with feature importance scores
    """
    
    print("\n" + "="*80)
    print("FEATURE SELECTION RESULTS")
    print("="*80)
    
    if n_components_to_use is None:
        n_components_to_use = results['n_components']
    
    loadings_df = results['loadings_df'].iloc[:, :n_components_to_use]
    
    # Calculate overall importance as sum of absolute loadings across components
    feature_importance = loadings_df.abs().sum(axis=1).sort_values(ascending=False)
    
    # Also calculate max absolute loading (most influential in any single PC)
    max_loading = loadings_df.abs().max(axis=1).sort_values(ascending=False)
    
    # Combine into DataFrame
    feature_importance_df = pd.DataFrame({
        'Feature': feature_importance.index,
        'Total_Importance': feature_importance.values,
        'Max_Loading': max_loading.loc[feature_importance.index].values
    }).reset_index(drop=True)
    
    # Select top features per component
    selected_features_set = set()
    
    for i in range(n_components_to_use):
        pc_name = f'PC{i+1}'
        top_features = loadings_df[pc_name].abs().nlargest(top_n_per_component).index.tolist()
        selected_features_set.update(top_features)
        
        print(f"\n{pc_name} - Top {top_n_per_component} features:")
        for j, feat in enumerate(top_features, 1):
            loading = loadings_df.loc[feat, pc_name]
            print(f"  {j:2d}. {feat:40s} (loading: {loading:7.4f})")
    
    selected_features = sorted(list(selected_features_set))
    
    print("\n" + "="*80)
    print(f"TOTAL SELECTED FEATURES: {len(selected_features)}")
    print("="*80)
    
    return selected_features, feature_importance_df


# ============================================================================
# EXPORT RESULTS
# ============================================================================

def export_results(results, feature_importance_df, selected_features, 
                   output_dir='./pca_results'):
    """
    Export PCA results to CSV files
    """
    
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\nExporting results to: {output_dir}/")
    
    # 1. Explained variance
    variance_df = pd.DataFrame({
        'Component': [f'PC{i+1}' for i in range(results['n_components'])],
        'Explained_Variance_Ratio': results['explained_variance_ratio'],
        'Cumulative_Variance': results['cumulative_variance']
    })
    variance_df.to_csv(f'{output_dir}/explained_variance.csv', index=False)
    print(f"  ✓ explained_variance.csv")
    
    # 2. Full loadings matrix
    results['loadings_df'].to_csv(f'{output_dir}/loadings_matrix.csv')
    print(f"  ✓ loadings_matrix.csv")
    
    # 3. Feature importance
    feature_importance_df.to_csv(f'{output_dir}/feature_importance.csv', index=False)
    print(f"  ✓ feature_importance.csv")
    
    # 4. Selected features
    selected_df = pd.DataFrame({'Selected_Features': selected_features})
    selected_df.to_csv(f'{output_dir}/selected_features.csv', index=False)
    print(f"  ✓ selected_features.csv")
    
    # 5. Summary report
    with open(f'{output_dir}/pca_summary_report.txt', 'w') as f:
        f.write("="*80 + "\n")
        f.write("FLEMMS PCA ANALYSIS SUMMARY REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Number of components: {results['n_components']}\n")
        f.write(f"Total explained variance: {results['cumulative_variance'][-1]*100:.2f}%\n")
        f.write(f"Number of original features: {len(results['feature_names'])}\n")
        f.write(f"Number of selected features: {len(selected_features)}\n")
        f.write(f"Reduction rate: {(1 - len(selected_features)/len(results['feature_names']))*100:.1f}%\n\n")
        
        f.write("Variance by component:\n")
        for i, var in enumerate(results['explained_variance_ratio'][:10], 1):
            f.write(f"  PC{i}: {var*100:.2f}%\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("Top 20 Most Important Features (by total loading):\n")
        f.write("="*80 + "\n")
        for i, row in feature_importance_df.head(20).iterrows():
            f.write(f"{i+1:3d}. {row['Feature']:45s} " +
                    f"(importance: {row['Total_Importance']:.4f})\n")
    
    print(f"  ✓ pca_summary_report.txt")
    print(f"\n✅ All results exported successfully!")


# ============================================================================
# MAIN EXECUTION FUNCTION
# ============================================================================

def run_flemms_pca(rtf1_file, rtf2_file, rtf3_file, member_file,
                   n_components=None, variance_threshold=0.95,
                   top_n_per_component=10, output_dir='./pca_results'):
    """
    Main function to run complete PCA analysis pipeline
    
    Parameters:
    -----------
    rtf1_file, rtf2_file, rtf3_file, member_file : str
        Paths to FLEMMS CSV files
    n_components : int, optional
        Number of PCA components (default: auto-select based on variance)
    variance_threshold : float
        Cumulative variance threshold for auto-selection
    top_n_per_component : int
        Number of top features to select per component
    output_dir : str
        Output directory for results
    
    Returns:
    --------
    results : dict
        Complete analysis results
    selected_features : list
        List of selected features for modeling
    """
    
    # Load data
    df_merged = load_flemms_data(rtf1_file, rtf2_file, rtf3_file, member_file)
    
    
    if len(df_merged) > 100000:
        print(f"Sampling 100,000 from {len(df_merged):,} rows for memory efficiency...")
        df_merged = df_merged.sample(n=100000, random_state=42)
    
    # Preprocess
    X_scaled, feature_names, df_metadata = preprocess_for_pca(
        df_merged, ALL_EXCLUDED_VARS
    )
    
    # Perform PCA
    pca, X_pca, results = perform_pca_analysis(
        X_scaled, feature_names, n_components, variance_threshold
    )
    
    # Visualize
    create_visualizations(pca, results, output_dir)
    
    # Feature selection
    selected_features, feature_importance_df = select_features_from_pca(
        results, n_components_to_use=None, top_n_per_component=top_n_per_component
    )
    
    # Export results
    export_results(results, feature_importance_df, selected_features, output_dir)
    
    # Add metadata to results
    results['df_metadata'] = df_metadata
    results['selected_features'] = selected_features
    results['feature_importance_df'] = feature_importance_df
    
    return results, selected_features


        # Run analysis
results, selected_features = run_flemms_pca(
    rtf1_file=rtf1_file,
    rtf2_file=rtf2_file,
    rtf3_file=rtf3_file,
    member_file=member_file,
    n_components=None,          # Auto-select based on variance
    variance_threshold=0.95,    # Explain 95% of variance
    top_n_per_component=10,     # Select top 10 features per PC
    output_dir='./flemms_pca_results'
)

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print(f"\nYou can now use the {len(selected_features)} selected features")
print("for your predictive modeling (XGBoost/LightGBM).")
print(f"\nResults saved to: ./flemms_pca_results/")


[1/5] Loading datasets...
  ✓ RTF1 loaded: (177656, 61)
  ✓ RTF2 loaded: (610590, 28)
  ✓ RTF3 loaded: (496770, 79)
  ✓ MEMBER loaded: (650424, 45)

  Standardizing column names to lowercase...
  📝 Standardized 61 column names in RTF1:
      'REG' → 'reg'
      'REG2' → 'reg2'
      'PRV' → 'prv'
      'HHID' → 'hhid'
      'URBANITY' → 'urbanity'
      ... and 56 more
  📝 Standardized 28 column names in RTF2:
      'REG' → 'reg'
      'REG2' → 'reg2'
      'PRV' → 'prv'
      'HHID' → 'hhid'
      'URBANITY' → 'urbanity'
      ... and 23 more
  📝 Standardized 79 column names in RTF3:
      'REG' → 'reg'
      'REG2' → 'reg2'
      'PRV' → 'prv'
      'HHID' → 'hhid'
      'URBANITY' → 'urbanity'
      ... and 74 more
  📝 Standardized 45 column names in MEMBER:
      'REG' → 'reg'
      'REG2' → 'reg2'
      'PRV' → 'prv'
      'HHID' → 'hhid'
      'URBANITY' → 'urbanity'
      ... and 40 more

[2/5] Merging datasets...
  ✓ After MEMBER + RTF1: (650424, 105)
  ✓ After + RTF2: (650424

In [32]:
"""
FLEMMS Data Diagnostic Script
==============================
Run this to check your data BEFORE running the full PCA analysis.
This will identify common issues that cause the 'tuple' error.

Usage:
    python flemms_diagnostic.py

Then update the file paths below to match your actual files.
"""

import pandas as pd
import sys


def check_file(filepath, file_label):
    """Check a single CSV file for common issues"""
    
    print(f"\n{'='*80}")
    print(f"Checking: {file_label}")
    print(f"{'='*80}")
    print(f"File path: {filepath}")
    
    try:
        # Try to load the file
        df = pd.read_csv(filepath)
        
        print(f"✅ File loaded successfully")
        print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
        print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        # Check for key columns
        print(f"\n📋 Column names (first 20):")
        for i, col in enumerate(df.columns[:20], 1):
            print(f"   {i:2d}. {col}")
        if len(df.columns) > 20:
            print(f"   ... and {len(df.columns) - 20} more columns")
        
        # Check for hhid
        hhid_variations = [col for col in df.columns if 'hhid' in col.lower()]
        if hhid_variations:
            print(f"\n✅ HHID column(s) found: {hhid_variations}")
            # Check for duplicates
            for hhid_col in hhid_variations:
                n_unique = df[hhid_col].nunique()
                n_total = len(df)
                print(f"   {hhid_col}: {n_unique:,} unique values out of {n_total:,} rows")
        else:
            print(f"\n⚠️  WARNING: No 'hhid' column found!")
            print(f"   Available columns: {df.columns.tolist()[:10]}")
        
        # Check for line number columns
        lno_variations = [col for col in df.columns if 'lno' in col.lower()]
        if lno_variations:
            print(f"\n✅ Line number column(s) found: {lno_variations}")
        
        # Check for missing values
        missing_counts = df.isnull().sum()
        cols_with_missing = missing_counts[missing_counts > 0]
        if len(cols_with_missing) > 0:
            print(f"\n⚠️  Missing values in {len(cols_with_missing)} columns")
            print(f"   Total missing cells: {missing_counts.sum():,}")
            print(f"   Top 5 columns with missing data:")
            for col, count in cols_with_missing.nlargest(5).items():
                pct = (count / len(df)) * 100
                print(f"     - {col}: {count:,} ({pct:.1f}%)")
        else:
            print(f"\n✅ No missing values")
        
        # Check data types
        print(f"\n📊 Data types:")
        dtype_counts = df.dtypes.value_counts()
        for dtype, count in dtype_counts.items():
            print(f"   {dtype}: {count} columns")
        
        return df
        
    except FileNotFoundError:
        print(f"❌ ERROR: File not found!")
        print(f"   Make sure the path is correct: {filepath}")
        return None
    except Exception as e:
        print(f"❌ ERROR: {type(e).__name__}: {e}")
        return None


def check_merge_compatibility(df1, df2, name1, name2, merge_key='hhid'):
    """Check if two dataframes can be merged"""
    
    print(f"\n{'='*80}")
    print(f"Checking merge compatibility: {name1} + {name2}")
    print(f"{'='*80}")
    
    if df1 is None or df2 is None:
        print(f"❌ Cannot check merge - one or both files failed to load")
        return
    
    # Check if merge key exists
    key_lower = merge_key.lower()
    
    # Find the actual column name (case-insensitive)
    key1 = None
    key2 = None
    
    for col in df1.columns:
        if col.lower() == key_lower:
            key1 = col
            break
    
    for col in df2.columns:
        if col.lower() == key_lower:
            key2 = col
            break
    
    if key1 is None:
        print(f"❌ {name1} is missing '{merge_key}' column")
        print(f"   Available columns: {df1.columns.tolist()[:10]}")
        return
    
    if key2 is None:
        print(f"❌ {name2} is missing '{merge_key}' column")
        print(f"   Available columns: {df2.columns.tolist()[:10]}")
        return
    
    print(f"✅ Both files have merge key column")
    print(f"   {name1}: '{key1}'")
    print(f"   {name2}: '{key2}'")
    
    # Check overlap
    set1 = set(df1[key1].dropna().unique())
    set2 = set(df2[key2].dropna().unique())
    
    overlap = set1 & set2
    only_in_1 = set1 - set2
    only_in_2 = set2 - set1
    
    print(f"\n📊 Merge key overlap:")
    print(f"   {name1} unique values: {len(set1):,}")
    print(f"   {name2} unique values: {len(set2):,}")
    print(f"   Overlapping values: {len(overlap):,}")
    
    if len(overlap) == 0:
        print(f"\n❌ WARNING: NO OVERLAP! These files cannot be merged.")
        print(f"   Values only in {name1}: {len(only_in_1):,}")
        print(f"   Values only in {name2}: {len(only_in_2):,}")
    elif len(overlap) < min(len(set1), len(set2)) * 0.5:
        print(f"\n⚠️  WARNING: Low overlap ({len(overlap)/min(len(set1), len(set2))*100:.1f}%)")
        print(f"   Values only in {name1}: {len(only_in_1):,}")
        print(f"   Values only in {name2}: {len(only_in_2):,}")
    else:
        print(f"   ✅ Good overlap!")
        if only_in_1:
            print(f"   Values only in {name1}: {len(only_in_1):,}")
        if only_in_2:
            print(f"   Values only in {name2}: {len(only_in_2):,}")


def main():
    """Main diagnostic routine"""
    
    print("\n" + "="*80)
    print("FLEMMS DATA DIAGNOSTIC TOOL")
    print("="*80)
    print("\nThis script will check your FLEMMS files for common issues.")
    print("Update the file paths below before running.\n")
    
    # ========================================================================
    # UPDATE THESE PATHS TO MATCH YOUR FILES
    # ========================================================================
    
    rtf1_file = r"D:\UP\CAPSTONE\Volume1\FLEMMS PUF 2024 Volume1 - RTF1.CSV"
    rtf2_file = r"Volume1/FLEMMS PUF 2024 Volume1 - RTF2.CSV"
    rtf3_file = r"D:\UP\CAPSTONE\Volume2\FLEMMS PUF 2024 Volume2 - RTF3.CSV"
    member_file = r"D:\UP\CAPSTONE\Volume1\FLEMMS PUF 2024 Volume1 - MEMBER.CSV"
    
    # ========================================================================
    
    # Check if paths are still default
    if 'path/to' in rtf1_file:
        print("⚠️  WARNING: You need to update the file paths in this script!")
        print("   Edit the 'rtf1_file', 'rtf2_file', etc. variables")
        print("   Then run this script again.\n")
        
        # Try to find CSV files in current directory
        import os
        csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
        if csv_files:
            print(f"CSV files found in current directory:")
            for f in csv_files[:10]:
                print(f"   - {f}")
            if len(csv_files) > 10:
                print(f"   ... and {len(csv_files) - 10} more")
        
        return
    
    # Check each file
    df_rtf1 = check_file(rtf1_file, "RTF1 (Household Questions)")
    df_rtf2 = check_file(rtf2_file, "RTF2 (Literacy Indicators)")
    df_rtf3 = check_file(rtf3_file, "RTF3 (Mass Media)")
    df_member = check_file(member_file, "MEMBER (Member Record)")
    
    # Check merge compatibility
    if df_member is not None:
        if df_rtf1 is not None:
            check_merge_compatibility(df_member, df_rtf1, "MEMBER", "RTF1", "hhid")
        if df_rtf2 is not None:
            check_merge_compatibility(df_member, df_rtf2, "MEMBER", "RTF2", "hhid")
        if df_rtf3 is not None:
            check_merge_compatibility(df_member, df_rtf3, "MEMBER", "RTF3", "hhid")
    
    # Summary
    print("\n" + "="*80)
    print("DIAGNOSTIC SUMMARY")
    print("="*80)
    
    files_ok = 0
    if df_rtf1 is not None:
        files_ok += 1
        print(f"✅ RTF1: OK")
    else:
        print(f"❌ RTF1: FAILED")
    
    if df_rtf2 is not None:
        files_ok += 1
        print(f"✅ RTF2: OK")
    else:
        print(f"❌ RTF2: FAILED")
    
    if df_rtf3 is not None:
        files_ok += 1
        print(f"✅ RTF3: OK")
    else:
        print(f"❌ RTF3: FAILED")
    
    if df_member is not None:
        files_ok += 1
        print(f"✅ MEMBER: OK")
    else:
        print(f"❌ MEMBER: FAILED")
    
    print(f"\n{files_ok}/4 files loaded successfully")
    
    if files_ok == 4:
        print("\n🎉 All files loaded successfully!")
        print("   You should be able to run the PCA analysis now.")
    else:
        print("\n⚠️  Some files failed to load.")
        print("   Fix the issues above before running PCA analysis.")
    
    print("="*80 + "\n")



In [33]:
main()


FLEMMS DATA DIAGNOSTIC TOOL

This script will check your FLEMMS files for common issues.
Update the file paths below before running.


Checking: RTF1 (Household Questions)
File path: D:\UP\CAPSTONE\Volume1\FLEMMS PUF 2024 Volume1 - RTF1.CSV
✅ File loaded successfully
   Shape: 177,656 rows × 61 columns
   Memory: 141.98 MB

📋 Column names (first 20):
    1. REG
    2. REG2
    3. PRV
    4. HHID
    5. URBANITY
    6. PSU_F1
    7. BENEFICIARY_4PS
    8. CHILDREN_04
    9. ADULTS
   10. GUARDIAN
   11. CHILD_FACILITY
   12. BUILDING
   13. ROOF
   14. WALL
   15. FLOOR_FINISH
   16. FLOOR_MAIN
   17. TENURE
   18. ELECTRICITY
   19. HH_REF
   20. HH_AIRCON
   ... and 41 more columns

✅ HHID column(s) found: ['HHID']
   HHID: 177,656 unique values out of 177,656 rows

✅ No missing values

📊 Data types:
   int64: 53 columns
   object: 7 columns
   float64: 1 columns

Checking: RTF2 (Literacy Indicators)
File path: Volume1/FLEMMS PUF 2024 Volume1 - RTF2.CSV
✅ File loaded successfully
   